# XAI Pipeline — Chest X-ray pneumonia CNN

Grayscale chest radiographs, NORMAL vs PNEUMONIA (single sigmoid unit).

All logic lives in `model/` and `xai/`. Train the model first with `python train_pneumonia.py` from the repository root, then run this notebook top to bottom.

## 1 · Setup

In [ ]:
import os, sys, json
from collections import defaultdict
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from model.runtime import setup
from xai.common import replace2linear
from xai.runner import ALL_METRICS, run_evaluation
from xai.visualization import plot_training_history, plot_cam_layers, plot_attribution_maps

from tf_keras_vis.utils.scores import BinaryScore
from model.loaders.pneumonia import CLASSES, load_datasets, preprocess_input, process_grayscale, rebalance_splits
from xai import pneumonia
from xai.visualization import prepare_rgb_image

setup(42)

In [ ]:
DATA_DIR       = 'data/chest_xray'
REBALANCED_DIR = 'data/chest_xray_rebalanced'
MODEL_PATH     = 'models/pneumonia.keras'
HISTORY_PATH   = 'models/pneumonia_history.json'
CSV_OUT        = 'results/pneumonia_metrics.csv'
IMAGE_SIZE     = (244, 244)

## 2 · Data

In [ ]:
if not Path(REBALANCED_DIR).exists():
    rebalance_splits(DATA_DIR, REBALANCED_DIR)
raw_train_ds, val_ds, test_ds = load_datasets(REBALANCED_DIR, IMAGE_SIZE)
train_ds = raw_train_ds.map(process_grayscale)
class_names = CLASSES

## 3 · Model

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()
with open(HISTORY_PATH) as f:
    plot_training_history(json.load(f))

## 4 · Classification performance

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
y_true, y_pred = [], []
for imgs, lbls in test_ds:
    y_true.extend(lbls.numpy().reshape(-1).astype(int))
    y_pred.extend((model.predict(imgs, verbose=0).reshape(-1) > 0.5).astype(int))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix')
plt.tight_layout(); plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))

## 5 · XAI setup

In [ ]:
background = pneumonia.get_background_samples(train_ds, 100)
evaluator = pneumonia.XAIEvaluator(model, fractions=10, robustness_n=10, sensitivity_iters=5, selectivity_patches=10)
pneumonia.register_all(evaluator, background)
print('Explainers:', list(evaluator.explainers))

## 6 · Grad-CAM and Grad-CAM++ across layers

In [ ]:
img_path = sorted(Path(REBALANCED_DIR, 'test', 'PNEUMONIA').glob('*.jpeg'))[0]
img_array = tf.cast(preprocess_input(str(img_path), IMAGE_SIZE), tf.float32)
print('P(PNEUMONIA) =', float(model.predict(img_array, verbose=0)[0, 0]))
to_display = lambda img: prepare_rgb_image(np.asarray(img))
plot_cam_layers(model, img_array, BinaryScore(True), pneumonia.LAYER_INDICES,
                to_display(img_array[0]), replace2linear)

## 7 · Attribution maps for confident predictions

In [ ]:
VIZ_CLASSES = CLASSES

CONF_THRESH = 0.9
candidates = defaultdict(list)
for batch_imgs, _ in val_ds:
    batch_imgs = np.asarray(batch_imgs)
    probs = model.predict(batch_imgs, verbose=0).reshape(-1)
    pred_idxs = (probs > 0.5).astype(int)
    confs = np.where(pred_idxs == 1, probs, 1 - probs)
    for img, pred_idx, conf in zip(batch_imgs, pred_idxs, confs):
        cls_name = class_names[pred_idx]
        if cls_name in VIZ_CLASSES and not candidates[cls_name] and CONF_THRESH <= conf <= CONF_THRESH + 0.11:
            candidates[cls_name].append((img, conf))
    if all(candidates[c] for c in VIZ_CLASSES):
        break
print({c: [round(float(conf), 3) for _, conf in v] for c, v in candidates.items()})

In [ ]:
for cls_name, items in candidates.items():
    for img, conf in items:
        plot_attribution_maps(evaluator, img, class_names.index(cls_name), to_display(img),
                              list(evaluator.explainers), f'{cls_name} (confidence {conf:.2f})')

## 8 · Quantitative evaluation

In [ ]:
images = pneumonia.get_balanced_sample(test_ds, num_samples=10)
results = run_evaluation(evaluator, images, ALL_METRICS, None, CSV_OUT, binary=True)
results